# MLB injured-list stints, 2015–2019, from MLB transactions (keyed by MLBAM ID)

Builds an injured-list table for seasons before the FanGraphs injury workbook begins, with the **same
columns as the FanGraphs sheets**, so the v4 / v5 notebooks can use it unchanged.

**Source.** The MLB Stats API transactions endpoint
(`https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate=…&endDate=…`), which is what the
mlb.com transactions pages are built on. Each transaction carries the player's **MLBAM ID** (`person.id`,
the same number as in his mlb.com player link), the date, the teams and the text, e.g.
*"Los Angeles Dodgers placed LHP Clayton Kershaw on the 15-day disabled list retroactive to June 27, 2016.
Herniated disc."* Using the API avoids scraping the HTML pages.

**Steps**

1. Download transactions month by month (cached as JSON in `data/mlb_transactions/`).
2. Classify injured-list transactions from their text: **placed**, **transferred** (e.g. 15-day → 60-day),
   **activated / reinstated**; plus **released / elected free agency / retired**, which end a stint without a
   return. Both wordings are handled: "disabled list" (through 2018) and "injured list" (2019 on).
3. Build stints per MLBAM ID: placement → activation. The **IL date** is the retroactive date when given,
   otherwise the placement date (FanGraphs' "IL Retro Date"); the **return date** is the activation date.
4. **Carry-overs across seasons.** A stint still open at the end of a season gets a row in that season with
   no return date, and a row in the next season (IL date = that season's re-placement date if MLB issued
   one, otherwise opening day), with **"Injury / Surgery Date" = the original IL date**. The v4/v5 baseline
   window then ends at the original injury, as it does for FanGraphs carry-overs.
5. Add `key_fangraphs` from the Chadwick register (the v4/v5 pipeline keys pitchers by FanGraphs ID) and keep
   `key_mlbam` as the primary key.
6. **Validate** by running the same steps on 2021–2023 and comparing with the FanGraphs workbook.
7. Write a combined workbook: the FanGraphs sheets unchanged plus one sheet per new season.

Run top to bottom on a machine with internet access.

## 1. Settings

In [ ]:
from pathlib import Path
import json
import re
import time
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from IPython.display import display

WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE / "data"
TX_CACHE = DATA_DIR / "mlb_transactions"
TX_CACHE.mkdir(parents=True, exist_ok=True)
CHADWICK_CACHE = DATA_DIR / "chadwick_register_ids.csv"          # shared with the v4 / v5 notebooks
FANGRAPHS_WORKBOOK = WORKSPACE / "MLB Injuries Raw Data.xlsx"     # 2021-2026, FanGraphs
OUTPUT_WORKBOOK = WORKSPACE / "MLB Injuries Raw Data 2015-2026.xlsx"

BUILD_SEASONS = [2015, 2016, 2017, 2018, 2019]
VALIDATE_SEASONS = [2021, 2022, 2023]     # rebuilt from MLB transactions and compared with FanGraphs ([] to skip)
FETCH_RANGES = [("2014-01-01", "2019-12-31")]                     # 2014 so 2015 carry-overs are found
if VALIDATE_SEASONS:
    FETCH_RANGES.append((f"{min(VALIDATE_SEASONS) - 1}-01-01", f"{max(VALIDATE_SEASONS)}-12-31"))

TX_URL = "https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate={start}&endDate={end}"

# Opening days (domestic), used only for the IL date of carry-over rows with no spring re-placement.
OPENING_DAY = {2014: "2014-03-31", 2015: "2015-04-05", 2016: "2016-04-03", 2017: "2017-04-02",
               2018: "2018-03-29", 2019: "2019-03-28", 2020: "2020-07-23", 2021: "2021-04-01",
               2022: "2022-04-07", 2023: "2023-03-30", 2024: "2024-03-28", 2025: "2025-03-27"}
OPENING_DAY = {y: pd.Timestamp(d) for y, d in OPENING_DAY.items()}
PITCHER_TOKENS = {"RHP", "LHP", "P"}

## 2. Download transactions (cached by month)

One request per calendar month. Files already in `data/mlb_transactions/` are reused, so re-running only
downloads what is missing. The cell then prints the fields of a few raw records to confirm the structure.

In [ ]:
def month_starts(start, end):
    return pd.date_range(pd.Timestamp(start).replace(day=1), pd.Timestamp(end), freq="MS")


def fetch_month(m, retries=3, sleep_seconds=0.5):
    path = TX_CACHE / f"transactions_{m:%Y_%m}.json"
    if path.exists():
        return json.loads(path.read_text())
    end = (m + pd.offsets.MonthEnd(0))
    url = TX_URL.format(start=f"{m:%Y-%m-%d}", end=f"{end:%Y-%m-%d}")
    for attempt in range(1, retries + 1):
        try:
            time.sleep(sleep_seconds)
            with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60) as r:
                payload = json.loads(r.read().decode("utf-8"))
            path.write_text(json.dumps(payload))
            return payload
        except Exception as error:
            print(f"  {m:%Y-%m} attempt {attempt} failed: {error!r}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Could not download transactions for {m:%Y-%m}")


raw = []
for start, end in FETCH_RANGES:
    for m in month_starts(start, end):
        raw.extend(fetch_month(m).get("transactions", []))
print(f"Transactions downloaded: {len(raw):,}")
keys = pd.Series([k for t in raw[:2000] for k in t]).value_counts()
print("Fields present (first 2,000 records):", dict(keys))
for t in [t for t in raw if "list" in str(t.get("description", "")).lower()][:3]:
    print(json.dumps(t, indent=1)[:900])

## 3. Flatten and classify injured-list transactions

The MLBAM ID comes from `person.id`. The text decides the event type. Descriptions that mention the
injured/disabled list but match none of the patterns are listed for review at the end of the cell (rehab
assignments and similar notes are expected there).

In [ ]:
def team_name(t):
    for k in ("toTeam", "fromTeam"):
        if isinstance(t.get(k), dict) and t[k].get("name"):
            return t[k]["name"]
    return None


tx = pd.DataFrame([{
    "tx_id": t.get("id"), "mlbam": (t.get("person") or {}).get("id"),
    "name": (t.get("person") or {}).get("fullName"), "team": team_name(t),
    "date": t.get("date"), "effective_date": t.get("effectiveDate"), "resolution_date": t.get("resolutionDate"),
    "type_code": t.get("typeCode"), "type_desc": t.get("typeDesc"),
    "description": (t.get("description") or "").strip()} for t in raw])
tx = tx.dropna(subset=["mlbam"]).drop_duplicates("tx_id" if tx["tx_id"].notna().all() else None)
tx["mlbam"] = tx["mlbam"].astype(int)
for c in ["date", "effective_date", "resolution_date"]:
    tx[c] = pd.to_datetime(tx[c], errors="coerce")
tx["event_date"] = tx["date"].fillna(tx["effective_date"])

POS = r"(?P<pos>RHP|LHP|P|C|1B|2B|3B|SS|LF|CF|RF|OF|IF|INF|DH|UT)"
LIST = r"(?P<days>\d+)-day (?:disabled list|injured list|DL|IL)"
DATE_TXT = r"(?P<retro>[A-Z][a-z]{2,8}\.? \d{1,2}(?:, \d{4})?)"    # year sometimes omitted
RX = {
    "placed": re.compile(rf"\bplaced {POS} (?P<name>.+?) on the {LIST}(?: retroactive to {DATE_TXT})?[.,;]?\s*(?P<injury>.*)$"),
    "transferred": re.compile(rf"\btransferred {POS} (?P<name>.+?) (?:from the \d+-day (?:disabled list|injured list|DL|IL) )?"
                              rf"to the {LIST}(?: retroactive to {DATE_TXT})?[.,;]?\s*(?P<injury>.*)$"),
    "activated": re.compile(rf"\b(?:activated|reinstated) {POS} (?P<name>.+?) from the {LIST}"),
}
ENDERS = re.compile(r"\b(released|elected free agency|retired|became a free agent)\b", re.I)
LIST_WORDS = re.compile(r"(?:disabled|injured) list|\b\d+-day (?:DL|IL)\b")


def parse_date_text(s, event_date=None):
    """'June 27, 2016' / 'Sept. 1, 2019'; without a year, the most recent such date on or before the event."""
    if not isinstance(s, str):
        return pd.NaT
    s = s.replace(".", "").replace("Sept ", "Sep ").strip()
    if not re.search(r"\d{4}$", s):
        if event_date is None or pd.isna(event_date):
            return pd.NaT
        d = parse_date_text(f"{s}, {event_date.year}")
        return d - pd.DateOffset(years=1) if pd.notna(d) and d > event_date else d
    for fmt in ("%B %d, %Y", "%b %d, %Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            pass
    return pd.NaT


def classify(desc):
    for kind, rx in RX.items():
        m = rx.search(desc)
        if m:
            g = m.groupdict()
            injury = (g.get("injury") or "").strip().rstrip(".").strip()
            return kind, g["pos"], int(g["days"]), g.get("retro"), injury
    if ENDERS.search(desc):
        return "ended", None, None, pd.NaT, ""
    return None, None, None, pd.NaT, ""


parsed = tx["description"].map(classify)
tx["event"] = parsed.map(lambda p: p[0])
tx["pos"] = parsed.map(lambda p: p[1])
tx["list_days"] = parsed.map(lambda p: p[2])
tx["retro_date"] = [parse_date_text(r, d) for r, d in zip(parsed.map(lambda p: p[3]), tx["event_date"])]
tx["injury"] = parsed.map(lambda p: p[4])
events = tx.loc[tx["event"].notna()].sort_values(["mlbam", "event_date", "tx_id"]).reset_index(drop=True)

display(events["event"].value_counts().rename("transactions"))
unmatched = tx.loc[tx["event"].isna() & tx["description"].str.contains(LIST_WORDS)]
unmatched.to_csv(DATA_DIR / "mlb_transactions_unmatched_IL_text.csv", index=False)
print(f"IL-related descriptions not classified: {len(unmatched)} (saved for review). Most common openings:")
display(unmatched["description"].str.replace(r"^[A-Z][\w.' ]+? (?=[a-z])", "", regex=True)
        .str.split().str[:4].str.join(" ").value_counts().head(15))

## 4. Build stints per player (MLBAM ID)

Events are processed in date order for each player:

* **placed** opens a stint (IL date = retroactive date if given). If he is already on the IL, it is a
  re-placement: same stint, and if it falls in a new season it starts that season's row (spring
  re-placements of players who ended the previous season on the IL).
* **transferred** keeps the stint open (opens one if the placement was missed, flagged).
* **activated** closes it; return date = activation date.
* **released / free agency / retired** closes it with no return date.
* Open at the end of the data → no return date.

A stint spanning seasons is written as one row per season (see the introduction).

In [ ]:
def emit_rows(st, ret, status):
    """Split one stint into per-season rows."""
    rows = []
    years = sorted(set(range(st["il_date"].year, (ret.year if pd.notna(ret) else st["last_year"]) + 1)))
    for y in years:
        first = y == st["il_date"].year
        last = y == years[-1]
        il = st["il_date"] if first else st["season_start"].get(y, OPENING_DAY.get(y, pd.Timestamp(f"{y}-04-01")))
        rows.append({
            "mlbam": st["mlbam"], "Name": st["name"], "Team": st["team"], "pos_token": st["pos"],
            "Season": y, "IL Retro Date": il, "placement_date": st["placed_on"] if first else st["season_start"].get(y, pd.NaT),
            "Injury / Surgery Date": pd.NaT if first else st["il_date"],
            "Injury / Surgery": st["injury_by_year"].get(y) or st["injury"],
            "Return Date": ret if last else pd.NaT,
            "Status": status if last else "Season ended on IL",
            "list_days_max": st["days_max"], "opened_by": st["opened_by"]})
    return rows


rows, flags = [], {"activation without open stint": 0, "stint opened by transfer": 0, "re-placement while on IL": 0}
for mlbam, ev in events.groupby("mlbam", sort=False):
    st = None
    for _, e in ev.iterrows():
        d = e["event_date"]
        if e["event"] == "placed" or (e["event"] == "transferred" and st is None):
            if st is None:
                il = e["retro_date"] if pd.notna(e["retro_date"]) else d
                st = {"mlbam": mlbam, "name": e["name"], "team": e["team"], "pos": e["pos"], "il_date": il,
                      "placed_on": d, "injury": e["injury"], "injury_by_year": {il.year: e["injury"]},
                      "days_max": e["list_days"], "season_start": {}, "last_year": il.year,
                      "opened_by": e["event"]}
                if e["event"] == "transferred":
                    flags["stint opened by transfer"] += 1
            else:
                flags["re-placement while on IL"] += 1
                if d.year > st["last_year"]:
                    st["season_start"][d.year] = e["retro_date"] if pd.notna(e["retro_date"]) else d
                    if e["injury"]:
                        st["injury_by_year"][d.year] = e["injury"]
                st["days_max"] = max(st["days_max"] or 0, e["list_days"] or 0)
                st["injury"] = st["injury"] or e["injury"]
                st["team"] = e["team"] or st["team"]
            st["last_year"] = max(st["last_year"], d.year)
        elif e["event"] == "transferred":
            st["days_max"] = max(st["days_max"] or 0, e["list_days"] or 0)
            st["injury"] = st["injury"] or e["injury"]
            st["last_year"] = max(st["last_year"], d.year)
        elif e["event"] == "activated":
            if st is None:
                flags["activation without open stint"] += 1
                continue
            rows += emit_rows(st, d, "Activated")
            st = None
        elif e["event"] == "ended" and st is not None:
            rows += emit_rows(st, pd.NaT, "Released / free agent / retired while on IL")
            st = None
    if st is not None:
        rows += emit_rows(st, pd.NaT, "On IL at end of data")

stints = pd.DataFrame(rows)
stints["Pos"] = np.where(stints["pos_token"].isin(PITCHER_TOKENS), "P", stints["pos_token"])
print({k: v for k, v in flags.items()})
display(stints.groupby(["Season", "Pos"]).size().unstack(fill_value=0).loc[:, lambda d: d.columns.isin(["P", "C", "OF", "SS"])])
display(stints.loc[stints["Pos"] == "P"].groupby("Season")["Status"].value_counts().unstack(fill_value=0))

## 5. FanGraphs IDs and the FanGraphs-style table

`key_mlbam` stays the primary key. `key_fangraphs` is added from the Chadwick register because the v4/v5
pipeline keys pitchers by FanGraphs ID; pitchers without one are listed.

In [ ]:
def clean_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return (s[:-2] if s.endswith(".0") else s) or None


if CHADWICK_CACHE.exists():
    id_map = pd.read_csv(CHADWICK_CACHE, dtype=str)
else:
    url = "https://raw.githubusercontent.com/chadwickbureau/register/master/data/people-{}.csv"
    id_map = pd.concat([pd.read_csv(url.format(s), usecols=["key_mlbam", "key_fangraphs"], dtype=str).dropna()
                        for s in "0123456789abcdef"], ignore_index=True)
    id_map.to_csv(CHADWICK_CACHE, index=False)
mlbam_to_fg = dict(zip(id_map["key_mlbam"].map(clean_id), id_map["key_fangraphs"].map(clean_id)))

out = stints.copy()
out["key_mlbam"] = out["mlbam"].astype(str)
out["key_fangraphs"] = out["key_mlbam"].map(mlbam_to_fg)
out["URL"] = "https://www.mlb.com/player/" + out["key_mlbam"]
out["Eligible to Return"] = pd.NaT
out["Latest Update"] = ""
out["source"] = "MLB Stats API transactions"
COLS = ["Name", "URL", "Team", "Pos", "Injury / Surgery Date", "Injury / Surgery", "Status", "IL Retro Date",
        "Eligible to Return", "Return Date", "Latest Update", "key_fangraphs", "key_mlbam",
        "placement_date", "list_days_max", "opened_by", "source"]
out = out[COLS + ["Season"]].sort_values(["Season", "IL Retro Date", "Name"]).reset_index(drop=True)
no_fg = out.loc[(out["Pos"] == "P") & out["key_fangraphs"].isna()]
print(f"Pitcher rows without a FanGraphs ID: {len(no_fg)} of {(out['Pos'] == 'P').sum()}")
display(no_fg[["Season", "Name", "key_mlbam"]].head(20))

## 6. Validation against FanGraphs (2021–2023)

The same steps, applied to 2021–2023 transactions, should reproduce the FanGraphs workbook's pitcher rows.
A FanGraphs row counts as matched when the same pitcher (MLBAM ID) has a rebuilt row in that season with an
IL date within 3 days. Return dates are then compared for matched rows. The unmatched lists are saved so a
pattern (e.g. a wording the parser misses) can be spotted.

In [ ]:
if VALIDATE_SEASONS and FANGRAPHS_WORKBOOK.exists():
    fg_to_mlbam = {v: k for k, v in mlbam_to_fg.items() if v}
    val_rows, miss_fg, miss_tx = [], [], []
    for s in VALIDATE_SEASONS:
        fg = pd.read_excel(FANGRAPHS_WORKBOOK, sheet_name=str(s))
        fg = fg.loc[fg["Pos"].astype(str).str.strip().str.upper().isin({"RP", "SP", "SP/RP", "RP/SP", "P"})].copy()
        fg["fg_id"] = fg["key_fangraphs"].map(clean_id) if "key_fangraphs" in fg else None
        fg["mlbam"] = fg["fg_id"].map(fg_to_mlbam)
        fg["il"] = pd.to_datetime(fg["IL Retro Date"], errors="coerce")
        fg["ret"] = pd.to_datetime(fg["Return Date"], errors="coerce")
        fg = fg.dropna(subset=["mlbam", "il"])
        mine = out.loc[(out["Season"] == s) & (out["Pos"] == "P")].copy()
        mine["il"] = pd.to_datetime(mine["IL Retro Date"])
        mine["ret"] = pd.to_datetime(mine["Return Date"])
        m = fg.merge(mine[["key_mlbam", "il", "ret", "Injury / Surgery"]], left_on="mlbam", right_on="key_mlbam",
                     how="left", suffixes=("", "_tx"))
        m["gap"] = (m["il_tx"] - m["il"]).dt.days.abs()
        best = m.sort_values("gap").drop_duplicates(["mlbam", "il"])
        ok = best["gap"] <= 3
        ret_both = ok & best["ret"].notna() & best["ret_tx"].notna()
        val_rows.append({"season": s, "FanGraphs pitcher rows": len(best), "matched (IL date within 3 days)": f"{ok.mean():.1%}",
                         "return date exact (both present)": f"{(best.loc[ret_both, 'ret'] == best.loc[ret_both, 'ret_tx']).mean():.1%}",
                         "return date within 3 days": f"{((best.loc[ret_both, 'ret'] - best.loc[ret_both, 'ret_tx']).dt.days.abs() <= 3).mean():.1%}",
                         "return missing in one source only": int((ok & (best["ret"].isna() != best["ret_tx"].isna())).sum()),
                         "rebuilt pitcher rows": len(mine)})
        miss_fg.append(best.loc[~ok, ["Name", "Injury / Surgery", "il", "ret"]].assign(season=s))
        mk = mine.merge(fg[["mlbam", "il"]], left_on="key_mlbam", right_on="mlbam", how="left", suffixes=("", "_fg"))
        mk["gap"] = (mk["il_fg"] - mk["il"]).dt.days.abs()
        mk = mk.sort_values("gap").drop_duplicates(["key_mlbam", "il"])
        miss_tx.append(mk.loc[~(mk["gap"] <= 3), ["Name", "Injury / Surgery", "il", "ret", "Status"]].assign(season=s))
    validation = pd.DataFrame(val_rows)
    display(validation)
    pd.concat(miss_fg).to_csv(DATA_DIR / "validation_fangraphs_rows_not_rebuilt.csv", index=False)
    pd.concat(miss_tx).to_csv(DATA_DIR / "validation_rebuilt_rows_not_in_fangraphs.csv", index=False)
    print("Unmatched rows saved to data/validation_*.csv for review.")
else:
    print("Validation skipped (no VALIDATE_SEASONS or FanGraphs workbook not found).")

## 7. Write the combined workbook

The FanGraphs sheets are copied unchanged; one sheet per season in `BUILD_SEASONS` is added with the same
column names (plus `key_mlbam` and a few provenance columns). In the v4/v5 notebooks, point
`INJURY_WORKBOOK` at this file and add the new seasons to `SEASONS` / `TRAIN_SEASONS`.

In [ ]:
with pd.ExcelWriter(OUTPUT_WORKBOOK) as xw:
    if FANGRAPHS_WORKBOOK.exists():
        src = pd.ExcelFile(FANGRAPHS_WORKBOOK)
        for sheet in src.sheet_names:
            pd.read_excel(src, sheet_name=sheet).to_excel(xw, sheet_name=str(sheet), index=False)
    for s in BUILD_SEASONS:
        sheet = out.loc[out["Season"] == s].drop(columns="Season")
        sheet.to_excel(xw, sheet_name=str(s), index=False)
summary = out.loc[out["Season"].isin(BUILD_SEASONS)].groupby("Season").agg(
    IL_rows=("Name", "size"), pitcher_rows=("Pos", lambda p: int((p == "P").sum())),
    pitchers=("key_mlbam", "nunique"), with_return_date=("Return Date", lambda r: int(r.notna().sum())))
display(summary)
print("Wrote", OUTPUT_WORKBOOK)

## 8. Notes before using these seasons in v4 / v5

* **Check the validation table first.** Match rates in the high 90s on 2021–2023 mean the rebuilt rows can
  stand in for FanGraphs. Read the two unmatched files for any systematic miss before trusting 2015–2019.
* **No separate injury date.** MLB transactions give the IL date, not the date of injury or surgery, except
  that carry-overs get the original IL date. FanGraphs sometimes records an earlier injury/surgery date, so
  baselines for 2015–2019 may run a little closer to the injury. The 30-day-lagged baseline makes this moot.
* **2015 baselines and PITCHf/x.** The 365-day baseline for 2015 injuries reaches into 2014, which comes from
  PITCHf/x, not Statcast; its velocities are not directly comparable (measured at a different point).
  Either start the extended cohort in 2016 or restrict 2015 baselines to 2015 games.
* **Injury descriptions** come from the same text, so the arm / elbow patterns in v4/v5 apply as they are;
  check `injury_classification_review.csv` there again after adding these seasons.
* **2020** stays out (shortened season), as before.